# Radiomic Feature Extraction — Brain Metastasis MRI

This notebook extracts quantitative radiomic features from segmented
brain metastasis tumors, covering four feature categories: **shape,
location, intensity, and texture**. The resulting feature table is
intended for a downstream classification analysis relating these
imaging features to the primary tumor origin.

Input: preprocessed and segmented MRI scans (see the companion
preprocessing and segmentation notebooks in this project).

**Source and license:** Adapted from the official PyRadiomics example
notebook ([AIM-Harvard/pyradiomics](https://github.com/AIM-Harvard/pyradiomics/blob/master/notebooks/RadiomicsExample.ipynb),
BSD-3-Clause License). Modified for full-cohort batch processing, a
non-binary source mask format, and an added location feature category.


## Configuration

Only this cell needs to change to run the notebook in a different
environment.


In [ ]:
DATA_DIR = "data/preprocessed/step5_skull_stripped"
OUTPUT_DIR = "outputs"
EXCLUDED_PATIENTS = ["010038"]  # excluded: empty tumor mask

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Imports

In [ ]:
from radiomics import featureextractor
import numpy as np

## Load patient list

One image + one tumor mask per patient. Patient IDs are the source
dataset's own de-identified case codes.


In [ ]:
import glob

patient_ids = sorted(set(
    os.path.basename(f).replace("_img_step5.nii", "")
    for f in glob.glob(os.path.join(DATA_DIR, "*_img_step5.nii"))
))
patient_ids = [p for p in patient_ids if p not in EXCLUDED_PATIENTS]
print(f"{len(patient_ids)} patients")

## Extract features

Tumor masks are binarized before extraction (any non-zero value =
tumor), since the source masks encode individual lesion identity
rather than a simple binary label.

Feature categories extracted:
- **Shape** — volume, sphericity, elongation, surface area, etc.
- **Intensity** — mean, variance, skewness, kurtosis, etc.
- **Texture** — GLCM, GLSZM, GLRLM, GLDM, and NGTDM descriptors
- **Location** — tumor centroid in a shared template coordinate
  space (meaningful across patients since all scans were previously
  registered to a common anatomical template during preprocessing)


In [ ]:
import SimpleITK as sitk
import pandas as pd

extractor = featureextractor.RadiomicsFeatureExtractor()
all_features = []
failed_patients = []

for pid in patient_ids:
    try:
        img = sitk.ReadImage(os.path.join(DATA_DIR, f"{pid}_img_step5.nii"))
        mask = sitk.ReadImage(os.path.join(DATA_DIR, f"{pid}_mask_step5.nii"))
        mask_binary = sitk.Cast(mask > 0, sitk.sitkUInt8)

        result = extractor.execute(img, mask_binary)
        features = {k: v for k, v in result.items() if k.startswith("original_")}

        mask_arr = sitk.GetArrayFromImage(mask_binary)
        coords = np.argwhere(mask_arr > 0)
        centroid_voxel = coords.mean(axis=0)
        centroid_physical = mask_binary.TransformContinuousIndexToPhysicalPoint(
            [float(centroid_voxel[2]), float(centroid_voxel[1]), float(centroid_voxel[0])]
        )
        features["location_centroid_x"] = centroid_physical[0]
        features["location_centroid_y"] = centroid_physical[1]
        features["location_centroid_z"] = centroid_physical[2]
        features["patient_id"] = pid
        all_features.append(features)

    except Exception as e:
        print(f"Failed on {pid}: {e}")
        failed_patients.append(pid)

features_df = pd.DataFrame(all_features)
features_df = features_df[["patient_id"] + [c for c in features_df.columns if c != "patient_id"]]
print(f"Extracted features for {len(all_features)}/{len(patient_ids)} patients")
if failed_patients:
    print(f"Failed: {failed_patients}")

## Save results

In [ ]:
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "radiomic_features.csv")
features_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {features_df.shape[0]} patients x {features_df.shape[1]} columns")
features_df.head()

## Feature distribution across the cohort

In [ ]:
import matplotlib.pyplot as plt

key_features = ["original_shape_MeshVolume", "original_shape_Elongation",
                 "original_shape_Sphericity", "original_firstorder_Mean"]

fig, axes = plt.subplots(1, len(key_features), figsize=(16, 5))
for ax, feat in zip(axes, key_features):
    if feat in features_df.columns:
        ax.boxplot(features_df[feat].dropna())
        ax.set_title(feat.replace("original_", ""))
plt.suptitle(f"Feature distribution across {len(features_df)} patients")
plt.tight_layout()
plt.show()

## Protocol feature coverage

| Category | Covered | Source |
|---|---|---|
| Shape | Yes | PyRadiomics `original_shape_*` |
| Intensity | Yes | PyRadiomics `original_firstorder_*` |
| Texture | Yes | PyRadiomics GLCM/GLSZM/GLRLM/GLDM/NGTDM |
| Location | Yes | Added — tumor centroid in shared template space |

Edema-based features (from FLAIR/T2) are not yet included, since the
current pipeline processes a single MRI modality (T1 contrast-enhanced).


## Next steps

Merge this feature table with tumor origin labels (patient ID as key)
and evaluate whether these features carry predictive signal for
primary tumor origin.
